# 02 Clean GDELT Headlines

This notebook validates the fixed 2025 headline dataset, removes repeated headlines, checks ambiguous company mappings, and saves the final cleaned dataset. It does not assign sentiment labels.


In [11]:
from pathlib import Path
from urllib.parse import urlparse
import re
import unicodedata

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)


In [12]:
# Project-root detection supports execution from the project root or the `pipeline` directory.
working_directory = Path.cwd()
project_root = working_directory.parent if working_directory.name == "pipeline" else working_directory

raw_path = project_root / "data" / "raw" / "gdelt_headlines.csv"
processed_directory = project_root / "data" / "processed"
deduplicated_path = processed_directory / "gdelt_headlines_deduplicated.csv"
review_path = processed_directory / "headline_mapping_review.csv"
exclusions_path = processed_directory / "headline_mapping_exclusions.csv"
clean_path = processed_directory / "gdelt_headlines_clean.csv"

print("Project root:", project_root)
print("Raw data exists:", raw_path.exists())


Project root: c:\Users\PC\Desktop\Masters\09 Capstone Project - Dissertation\Week 40\DK-CoT_UK_Stocks
Raw data exists: True


In [13]:
raw_df = pd.read_csv(raw_path)

print(f"Raw headlines: {len(raw_df):,}")
display(raw_df["company_name"].value_counts())
raw_df.head()


Raw headlines: 18,797


company_name
HSBC Holdings plc     5480
Vodafone Group plc    3793
AstraZeneca plc       2854
GSK plc               1943
Rio Tinto plc         1737
National Grid plc     1714
Intertek Group plc     683
RELX plc               318
Smith & Nephew plc     165
Halma plc              110
Name: count, dtype: int64

,published_at_utc,url,ticker,company_name,headline_text
0,2025-04-30T08:02:09+00:00,https://www.marketscreener.com/quote/stock/SMITH-NEPHEW-PLC-9590181/,SN.L,Smith & Nephew plc,Smith & Nephew Plc Stock (SN.) - Quote London S.E.- MarketScreener
1,2025-04-30T08:02:21+00:00,https://www.zawya.com/en/business/banking-and-insurance/hsbc-sees-revenue-and-bad-loan-risk-from-tariff-fight-ifr-cl12jjjw,HSBA.L,HSBC Holdings plc,HSBC sees revenue and bad loan risk from tariff fight: IFR
2,2025-04-30T00:46:47+00:00,https://www.dailymail.co.uk/money/markets/article-14660189/Step-lose-jobs-US-AstraZeneca-boss-Pascal-Soriot-warns-European-pharma-firms.html,AZN.L,AstraZeneca plc,"Step up now or lose jobs to US, AstraZeneca boss Pascal Soriot warns European pharma firms"
3,2025-04-30T09:46:38+00:00,https://www.bordertelegraph.com/news/national/25126355.drugs-giant-gsk-well-positioned-absorb-potential-us-tariffs/,GSK.L,GSK plc,Drugs giant GSK 'well positioned' to absorb potential US tariffs
4,2025-04-30T08:31:38+00:00,https://www.dailyecho.co.uk/news/national/25126355.drugs-giant-gsk-well-positioned-absorb-potential-us-tariffs/,GSK.L,GSK plc,Drugs giant GSK 'well positioned' to absorb potential US tariffs


In [14]:
# Schema validation covers the expected columns, company universe, dates and required values.
DATA_COLUMNS = [
    "published_at_utc",
    "url",
    "ticker",
    "company_name",
    "headline_text",
]
EXPECTED_TICKERS = {
    "AZN.L", "HSBA.L", "GSK.L", "RIO.L", "VOD.L",
    "NG.L", "REL.L", "HLMA.L", "SN.L", "ITRK.L",
}

assert raw_df.columns.tolist() == DATA_COLUMNS
assert not raw_df[DATA_COLUMNS].isna().any().any()
assert set(raw_df["ticker"]) == EXPECTED_TICKERS

raw_df["_published_at"] = pd.to_datetime(
    raw_df["published_at_utc"],
    utc=True,
    errors="raise",
)

start_time = pd.Timestamp("2025-01-01", tz="UTC")
end_time = pd.Timestamp("2026-01-01", tz="UTC")
assert raw_df["_published_at"].between(start_time, end_time, inclusive="left").all()

print("Schema, company coverage, required values and 2025 dates are valid.")


Schema, company coverage, required values and 2025 dates are valid.


In [15]:
def normalize_headline(value):
    """Create a consistent title used only for duplicate detection."""

    normalized = unicodedata.normalize("NFKC", value)
    without_formatting_marks = "".join(
        character
        for character in normalized
        if unicodedata.category(character) != "Cf"
    )
    return re.sub(r"\s+", " ", without_formatting_marks).strip().casefold()


working_df = raw_df.copy()
working_df["normalized_headline"] = working_df["headline_text"].map(
    normalize_headline
)


In [16]:
# Chronological ordering allows keep='first' to preserve the earliest observation.
working_df = working_df.sort_values(
    ["_published_at", "ticker", "headline_text"]
).reset_index(drop=True)

duplicate_mask = working_df.duplicated(
    subset=["ticker", "normalized_headline"],
    keep="first",
)

duplicates_removed = int(duplicate_mask.sum())
deduplicated_df = working_df.loc[~duplicate_mask].copy().reset_index(drop=True)

assert not deduplicated_df.duplicated(
    subset=["ticker", "normalized_headline"]
).any()

print(f"Repeated rows removed: {duplicates_removed:,}")
print(f"Deduplicated headlines: {len(deduplicated_df):,}")


Repeated rows removed: 8,276
Deduplicated headlines: 10,521


In [17]:
deduplication_summary = pd.concat(
    [
        raw_df.groupby("ticker").size().rename("raw"),
        deduplicated_df.groupby("ticker").size().rename("deduplicated"),
    ],
    axis=1,
)
deduplication_summary["removed"] = (
    deduplication_summary["raw"] - deduplication_summary["deduplicated"]
)

display(deduplication_summary)


,raw,deduplicated,removed
ticker,,,
AZN.L,2854,1381,1473
GSK.L,1943,954,989
HLMA.L,110,89,21
HSBA.L,5480,2906,2574
ITRK.L,683,96,587
NG.L,1714,1029,685
REL.L,318,121,197
RIO.L,1737,1120,617
SN.L,165,112,53


## Company-mapping relevance

The mapping audit identified systematic false matches involving separate Vodafone entities, generic or foreign uses of “national grid”, unrelated uses of “Halma”, and generic directory or advertising pages. The deterministic rules below address these cases. Remaining uncertain Vodafone and National Grid matches are recorded in a review table, preserving the mapping audit.


In [18]:
NON_NEWS_PATTERNS = [
    ("generic_rns_index", r"\brns announcements\b"),
    ("generic_stock_quote", r"\bstock quote\b|\bstock\s*\([^)]+\)\s*-\s*quote london s\.e\.-\s*marketscreener"),
    ("generic_news_index", r"^[^|]{1,100}\bnews\s*\|\s*[^|]{0,100}\blatest news$"),
    ("advertising_market_page", r"\bmarket (?:size|research and growth forecast|growth opportunities|business boosting strategies)\b|\bmarket report and company analysis\b|\bby key companies\b"),
]

GENERIC_DIRECTORY_TITLES = {"intertek alchemy", "intertek - new food magazine"}

HALMA_UNRELATED = r"\bfadia halma\b|\bfrancisco halma\b|\bhalma tradition\b|\bhalma(?:'s|’s) initiative on water crisis\b"

VODAFONE_SEPARATE = r"\bvodafone[- ]idea\b|\bvodafone ukraine\b|\bvf ukraine\b"

VODAFONE_GROUP_TRANSACTION = r"\bvodafone (?:group )?(?:sells|closes)\b.*\b(?:indus towers|stake in (?:india's )?vodafone idea)\b|\bvodafone group\b.*\bvodafone (?:idea|ukraine)\b|\bvodafone (?:idea|ukraine)\b.*\bvodafone group\b"

VODAFONE_UNCERTAIN = r"\bagr\b|\bvodafone[- ]india\b|\bvi\b|\bvodafone (?:qatar|samoa|spain|oman|italia|italy)\b"

NATIONAL_GRID_SEPARATE = r"\bnational grid eso\b|\bnational energy system operator\b|\bneso\b"

NATIONAL_GRID_FOREIGN = r"\b(?:nigeria|nigerian|tcn|tinubu|adelabu|peter obi|shettima|niso|kainji|lagos|abuja|bauchi|kaduna|kano|osun|benin-omotosho|ogoniland|ghana|ghanaian|gonja|malawi|zambia|ethiopia|ethiopian|kenya|kenyan|uganda|ugandan|sri lanka|pakistan|pakistani|sindh|nepra|k-electric|philippines|philippine|maharlika|eskom|south africa|azerbaijan|garabagh|zangazur|oman|masirah|vietnam|viet nam|nhơn trạch|flamanville|petromidia|romania|myagdi|nepal|cuba|cuban)\b|\bireland(?:'s)? (?:electricity|national grid)\b"

NATIONAL_GRID_COMPANY = r"\bnational grid (?:plc|transco|shares?|stock|share price|dividend|profit|earnings|results|revenue|investors?|analysts?|chief|ceo|cfo|chair|boss|customers?|crews?|bills?|billing|employees?|workers?|warns?|appoints?|acquires?|sells?|plans?|invests?|launches?)\b|\b(?:nyse\s*:\s*ngg|lon\s*:\s*ng|national grid\s*\(ngg\))\b|\b(?:massachusetts|rhode island|new york|upstate|buffalo|syracuse|albany|worcester|nantucket|brooklyn|long island)\b.*\bnational grid\b|\bnational grid\b.*\b(?:massachusetts|rhode island|new york|upstate|buffalo|syracuse|albany|worcester|nantucket|brooklyn|long island)\b"

NATIONAL_GRID_GENERIC = r"\b(?:connect(?:ed|s|ing)?|link(?:ed|s|ing)?|feed(?:s|ing)?|inject(?:s|ed|ing)?|supply(?:ing|ies|ied)?|add(?:s|ed|ing)?|export(?:s|ed|ing)?)\b.{0,100}\b(?:to|into)\s+(?:the\s+)?national grid\b|\b(?:to|into|off)\s+(?:the\s+)?national grid\b|\bnational grid (?:collapse|operator|system|stability|demand|control centres?|connection)\b"

ADVERTISING_UNCERTAIN = r"\b(?:comprehensive report|market forecast|competitive landscape|industry report)\b"


In [19]:
def classify_company_mapping(row):
    """Return retain, exclude or review using the documented mapping rules."""

    title = row["headline_text"]
    ticker = row["ticker"]

    if ticker == "VOD.L" and re.search(VODAFONE_SEPARATE, title, re.IGNORECASE):
        if re.search(VODAFONE_GROUP_TRANSACTION, title, re.IGNORECASE):
            return "retain", "vodafone_group_transaction"
        return "exclude", "vodafone_separate_entity"

    if ticker == "HLMA.L" and re.search(HALMA_UNRELATED, title, re.IGNORECASE):
        return "exclude", "halma_name_collision"

    if ticker == "NG.L":
        if re.search(NATIONAL_GRID_SEPARATE, title, re.IGNORECASE):
            return "exclude", "national_grid_separate_operator"
        if re.search(NATIONAL_GRID_FOREIGN, title, re.IGNORECASE):
            return "exclude", "foreign_national_grid"

    if row["normalized_headline"] in GENERIC_DIRECTORY_TITLES:
        return "exclude", "generic_directory_page"

    for rule, pattern in NON_NEWS_PATTERNS:
        if re.search(pattern, title, re.IGNORECASE):
            return "exclude", rule

    if ticker == "VOD.L" and re.search(VODAFONE_UNCERTAIN, title, re.IGNORECASE):
        return "review", "uncertain_vodafone_entity"

    if ticker == "NG.L":
        if re.search(NATIONAL_GRID_COMPANY, title, re.IGNORECASE):
            return "retain", "national_grid_company_context"
        # Lower-case "national grid" without company context is treated as a generic infrastructure reference.
        if re.search(r"\bnational grid\b", title) or re.search(NATIONAL_GRID_GENERIC, title, re.IGNORECASE):
            return "review", "uncertain_national_grid_reference"

    if re.search(ADVERTISING_UNCERTAIN, title, re.IGNORECASE):
        return "review", "uncertain_advertising_page"

    return "retain", "no_exclusion_signal"


In [20]:
mapping_results = deduplicated_df.apply(
    classify_company_mapping,
    axis=1,
    result_type="expand",
)
mapping_results.columns = ["mapping_decision", "filter_rule"]
deduplicated_df[["mapping_decision", "filter_rule"]] = mapping_results

# Two reviewed headlines were confirmed as false matches for the selected listed companies.
titles = deduplicated_df["headline_text"]
manual_exclusion_mask = (
    (
        (deduplicated_df["ticker"] == "GSK.L")
        & titles.str.contains("GSK Nigeria, Union Homes", case=False, regex=False, na=False)
    )
    | (
        (deduplicated_df["ticker"] == "RIO.L")
        & titles.str.contains("Graphene Aluminium-Ion Battery", case=False, regex=False, na=False)
        & titles.str.contains("IT Business Net", case=False, regex=False, na=False)
    )
)
deduplicated_df.loc[manual_exclusion_mask, "mapping_decision"] = "exclude"
deduplicated_df.loc[manual_exclusion_mask, "filter_rule"] = "manual_mapping_check"

display(deduplicated_df["mapping_decision"].value_counts())


mapping_decision
retain     8969
exclude    1299
review      253
Name: count, dtype: int64

In [21]:
# Final mapping decisions for the uncertain rows.
review_df = deduplicated_df.loc[
    deduplicated_df["mapping_decision"] == "review"
].sort_values(["_published_at", "ticker", "headline_text"]).reset_index(drop=True)

reviewed_relevant_indices = [
    0, 3, 4, 6, 7, 8, 9, 10, 13, 15, 21, 22, 27, 31,
    55, 56, 57, 59, 68, 90, 92, 93, 94, 95, 96, 97,
    119, 149, 165, 166, 251, 252,
]

assert reviewed_relevant_indices[-1] < len(review_df)
review_df["mapping_relevant"] = "no"
review_df.loc[reviewed_relevant_indices, "mapping_relevant"] = "yes"

display(review_df["mapping_relevant"].value_counts())
display(
    review_df.loc[
        review_df["mapping_relevant"] == "yes",
        ["ticker", "headline_text", "mapping_relevant"],
    ]
)


mapping_relevant
no     221
yes     32
Name: count, dtype: int64

,ticker,headline_text,mapping_relevant
0,NG.L,Is it time to tap into National Grid?,yes
3,VOD.L,Vodafone Group Completes Sale Of Vodafone Italy,yes
4,VOD.L,Vodafone Group Completes Sale Of Vodafone Italy - Quick Facts,yes
6,VOD.L,Swisscom seals Vodafone Italia deal in 'new era for Italian telecoms',yes
7,VOD.L,Swisscom completes acquisition of Vodafone Italia,yes
8,VOD.L,Swisscom completes takeover of Vodafone Italia,yes
9,VOD.L,Regulators approve Swisscom-Vodafone Italia merger,yes
10,VOD.L,FIVE at FIVE: FTSE 100 rallies; Vodafone Italy sale; Tesla underwhelms; House prices jump; Revolution Beauty surges,yes
13,VOD.L,Sale of Vodafone Italy for €8 billion completes,yes
15,VOD.L,Swisscom completes acquisition of Vodafone Italia – SatNews,yes


In [22]:
# The complete set of uncertain mapping decisions is displayed for verification.
display(
    review_df[
        ["ticker", "company_name", "headline_text", "mapping_relevant"]
    ]
)


,ticker,company_name,headline_text,mapping_relevant
0,NG.L,National Grid plc,Is it time to tap into National Grid?,yes
1,VOD.L,Vodafone Group plc,Swisscom revises down earnings outlook after Vodafone Italia acquisition complete,no
2,VOD.L,Vodafone Group plc,Swisscom Cuts Earnings Guidance on Early Closing of Vodafone Italia Deal,no
3,VOD.L,Vodafone Group plc,Vodafone Group Completes Sale Of Vodafone Italy,yes
4,VOD.L,Vodafone Group plc,Vodafone Group Completes Sale Of Vodafone Italy - Quick Facts,yes
...,...,...,...,...
248,NG.L,National Grid plc,Panasian Power connects 5MW solar unit to national grid,no
249,VOD.L,Vodafone Group plc,"Vodafone AGR dues: Cabinet likely approved relief, telco gets 5-year moratorium, reports CNBC-Awaaz",no
250,VOD.L,Vodafone Group plc,"Vodafone AGR Dues: Cabinet Approves Relief Package, Freezes AGR Dues At Rs 87,695 Crore, Says Report",no
251,VOD.L,Vodafone Group plc,Vodafone reaches agreement with Vi on CLAM | Company Announcement,yes


In [23]:
def get_source(url):
    """Return the publisher domain without a leading www."""

    hostname = urlparse(url).hostname
    assert hostname is not None
    return hostname.removeprefix("www.")


automatic_retained_df = deduplicated_df.loc[
    deduplicated_df["mapping_decision"] == "retain",
    DATA_COLUMNS,
].copy()
automatic_retained_df["mapping_confidence"] = "high"

reviewed_relevant_df = review_df.loc[
    review_df["mapping_relevant"] == "yes",
    DATA_COLUMNS,
].copy()
reviewed_relevant_df["mapping_confidence"] = "reviewed"

clean_df = pd.concat(
    [automatic_retained_df, reviewed_relevant_df],
    ignore_index=True,
).sort_values(["published_at_utc", "ticker", "headline_text"]).reset_index(drop=True)
clean_df["source"] = clean_df["url"].map(get_source)

automatic_exclusions_df = deduplicated_df.loc[
    deduplicated_df["mapping_decision"] == "exclude",
    [*DATA_COLUMNS, "filter_rule"],
].copy()
reviewed_exclusions_df = review_df.loc[
    review_df["mapping_relevant"] == "no",
    DATA_COLUMNS,
].copy()
reviewed_exclusions_df["filter_rule"] = "manual_review_no"

exclusions_df = pd.concat(
    [automatic_exclusions_df, reviewed_exclusions_df],
    ignore_index=True,
).sort_values(["published_at_utc", "ticker", "headline_text"]).reset_index(drop=True)

assert len(clean_df) + len(exclusions_df) == len(deduplicated_df)
assert not clean_df.duplicated(["ticker", "headline_text"]).any()
assert set(clean_df["ticker"]) == EXPECTED_TICKERS

print(f"Retained clean headlines: {len(clean_df):,}")
print(f"Excluded headlines: {len(exclusions_df):,}")


Retained clean headlines: 9,001
Excluded headlines: 1,520


In [24]:
processed_directory.mkdir(parents=True, exist_ok=True)

deduplicated_df[DATA_COLUMNS].to_csv(deduplicated_path, index=False)
review_df[[*DATA_COLUMNS, "mapping_relevant"]].to_csv(review_path, index=False)
exclusions_df.to_csv(exclusions_path, index=False)
clean_df.to_csv(clean_path, index=False)

stage_summary = pd.DataFrame(
    {
        "stage": ["raw", "deduplicated", "final clean", "excluded"],
        "rows": [len(raw_df), len(deduplicated_df), len(clean_df), len(exclusions_df)],
    }
)
display(stage_summary)

print("Saved:")
print(deduplicated_path)
print(review_path)
print(exclusions_path)
print(clean_path)


,stage,rows
0,raw,18797
1,deduplicated,10521
2,final clean,9001
3,excluded,1520


Saved:
c:\Users\PC\Desktop\Masters\09 Capstone Project - Dissertation\Week 40\DK-CoT_UK_Stocks\data\processed\gdelt_headlines_deduplicated.csv
c:\Users\PC\Desktop\Masters\09 Capstone Project - Dissertation\Week 40\DK-CoT_UK_Stocks\data\processed\headline_mapping_review.csv
c:\Users\PC\Desktop\Masters\09 Capstone Project - Dissertation\Week 40\DK-CoT_UK_Stocks\data\processed\headline_mapping_exclusions.csv
c:\Users\PC\Desktop\Masters\09 Capstone Project - Dissertation\Week 40\DK-CoT_UK_Stocks\data\processed\gdelt_headlines_clean.csv


## Result

The cleaned dataset used for subsequent price alignment and sentiment analysis is saved as `data/processed/gdelt_headlines_clean.csv`. Decisions for uncertain mappings are recorded in `headline_mapping_review.csv` and remain visible in this notebook, preserving the review trail.
